# dfthcsos: (8, 143, C) with C = 143 and C = 200 (commit mode)

Run it with **Save Version → Save & Run All (Commit)**. After every step the best verified result so far is packaged
as `/kaggle/working/submission.zip`.

Inputs (right panel, **+ Add Input**): the competition, the *Verifiable Circuit Compilation Starter Kit* dataset, the
`dfthcsos-toolkit` dataset (v5, unchanged) and, optionally, **the output of version 6 of this notebook** (the v4 run, with
`runs/S100.npz`). Settings: **GPU** accelerator, **Internet** on.

What v4 showed: more copies lower the error (C = 80: 25.2 mHa, C = 100: 23.0 mHa, continuous) at lambda_eff ~ 74.
The 13-bit build passes once the continuous error is below ~18.5 mHa.

Plan:
1. C = 143, warm-started from S100 (the 43 new copies start with tiny weights, so the error is unchanged). Without the v4 output it starts from scratch.
2. C = 200 from the eigendecomposition-based start, in a single 200k-epoch stage.
3. Weight refit (QAT) at 12 and 13 bits of each result that did not pass.

A warm restart uses a small learning rate (3e-5): in v4 the restart at 1e-4 threw the error from 24 to 390 mHa
and stage 2 only recovered what was lost.


In [ ]:
# Settings. EPOCH_SCALE < 1 shortens every run (only for quick tests); keep it at 1.0 for real runs.
EPOCH_SCALE = 1.0

In [ ]:
# 1. Locate the inputs and prepare the work folders
import functools, json, os, shutil, subprocess, sys, traceback, zipfile
from pathlib import Path

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')

def find(pattern, what):
    hits = list(INPUT.glob(pattern))
    if not hits:
        raise FileNotFoundError(f'{what} not found: add it with "+ Add Input" (right panel).')
    return hits[0]

OFFICIAL = INPUT / 'competitions' / 'google-verifiable-quantum-circuit-compilation'
COMPETITION = OFFICIAL if (OFFICIAL / 'verifiers').is_dir() else \
    find('**/verifiers/dfthcsos/verify.py', 'the competition data').parents[2]
KIT_SRC = find('**/starter_kit/starter', 'the starter kit').parent
TOOLKIT_SRC = find('**/init_8_143_44*', 'the dfthcsos-toolkit dataset (v5, with init_8_143_44.npz)').parent
print('competition :', COMPETITION)
print('starter kit :', KIT_SRC)
print('toolkit     :', TOOLKIT_SRC)

# The toolkit is copied to a writable folder.
TOOLKIT = WORK / 'toolkit'
shutil.copytree(TOOLKIT_SRC, TOOLKIT, dirs_exist_ok=True)

# The trainer looks for the FCIDUMP and the fixed shifts in the folder above starter_kit/.
KIT_PARENT = WORK / 'kit'; KIT_PARENT.mkdir(exist_ok=True)
for link, target in [('starter_kit', KIT_SRC),
                     ('FeMoCo76o113e.fcidump', COMPETITION / 'verifiers/dfthcsos/FeMoCo76o113e.fcidump'),
                     ('FeMoCo76o113e.fixed_shifts.npz', COMPETITION / 'verifiers/dfthcsos/FeMoCo76o113e.fixed_shifts.npz')]:
    if not (KIT_PARENT / link).exists():
        (KIT_PARENT / link).symlink_to(target)

os.environ.update({'COMPETITION_DIR': str(COMPETITION),
                   'KIT_DIR': str(KIT_PARENT / 'starter_kit'),
                   'WORK_DIR': str(WORK)})
print('ok')

In [ ]:
# 2. Check the GPU and the libraries
import importlib.util
if importlib.util.find_spec('optax') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optax'], check=True)
import jax
print('JAX backend:', jax.default_backend())  # should be 'gpu'

In [ ]:
# 3. Load the toolkit and define the helpers
import numpy as np

sys.path.insert(0, str(TOOLKIT))
os.chdir(TOOLKIT)
import run_experiments as RX
import starter.task1_factorization.train_df_thc_qaggle as TQ
from dfthc import train_df_thc_sga as TS
from qat import qat as _qat

REF = RX.lab.ref()
SORTED_SEED = TOOLKIT / 'FeMoCo76o113e_65_60_60_sorted.npz'
FACTORIZATION_KEYS = ('U', 'W', 'H2_I', 'H1_I', 'Hsym_lower')


def log(record):
    """Print a result and append it to results.jsonl."""
    print(json.dumps(record, indent=1, default=float))
    with open(WORK / 'results.jsonl', 'a') as f:
        f.write(json.dumps(record, default=float) + '\n')


def package():
    """Write submission.zip from the best verified result so far (reference circuits for invsqrt and dqi)."""
    results = WORK / 'results.jsonl'
    rows = [json.loads(line) for line in open(results)] if results.exists() else []
    for r in rows:
        print(f"{r['name']:14s} status={str(r.get('status')):20s} brot={r.get('brot')} "
              f"improvement={r.get('improvement', float('nan')):.4f}")
    passed = [r for r in rows if r.get('status') == 'PASS']
    if not passed:
        print('No verified result yet: submission.zip not written.')
        return
    best = max(passed, key=lambda r: r['improvement'])
    sample = COMPETITION / 'sample_submission'
    with zipfile.ZipFile(WORK / 'submission.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for name in ('invsqrt.kmb', 'dqi.kmb'):
            zf.write(sample / name, name)
        for path in sorted(Path(best['folder']).iterdir()):
            zf.write(path, f'dfthcsos/{path.name}')
    print(f"submission.zip <- {best['name']} (improvement x{best['improvement']:.4f})")


def seed_npz(name, search_root=INPUT, out_dir=WORK / 'seeds'):
    """Return a .npz for factorization `name`, rebuilding it if Kaggle unpacked it into a folder of .npy files."""
    files = [p for p in Path(search_root).glob(f'**/{name}.npz') if p.is_file()]
    if files:
        arrays = dict(np.load(files[0]))
    else:
        folders = [p for p in Path(search_root).glob(f'**/{name}') if (p / 'U.npy').is_file()]
        if not folders:
            raise FileNotFoundError(f'{name}: neither {name}.npz nor a folder with U.npy under {search_root}')
        arrays = {key: np.load(folders[0] / f'{key}.npy') for key in FACTORIZATION_KEYS}
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    path = Path(out_dir) / f'{name}.npz'
    np.savez(path, **{key: arrays[key] for key in FACTORIZATION_KEYS})
    print('seed:', path, arrays['U'].shape)
    return path


def build_only(name, source, brot):
    """Build and verify a trained factorization at `brot` bits (with the weight refit), log it and repackage."""
    record = {'name': name, 'source': str(source)}
    try:
        folder = WORK / 'builds' / f'{name}_b{brot}' / 'dfthcsos'
        m = RX.lab.build_folder(source, folder, brot=brot, refit=True, r=REF)
        record[f'b{brot}_local'] = {k: m[k] for k in ('error_1', 'error_2', 'Lambda', 'E_gap', 'lambda_eff')}
        summary = RX.verify(folder)
        record.update(brot=brot, folder=str(folder), status=summary.get('status'),
                      cost=summary.get('score'), error=summary.get('error'))
        if summary.get('status') == 'PASS':
            record['improvement'] = RX.BASELINE_COST / summary['score']
    except Exception:
        traceback.print_exc()
        record['status'] = 'ERROR'
    log(record)
    package()


def run_qat(name, source, brot, epochs=20_000):
    """Freeze the vectors at their brot-bit values, retrain only the weights, build, verify and repackage."""
    try:
        _qat(name, source, RX=RX, TQ=TQ, TS=TS, REF=REF, WORK=WORK, brot=brot,
             epochs=max(20, int(epochs * EPOCH_SCALE)))
    except Exception:
        traceback.print_exc()
    package()


def train(name, seed=None, lr_final=1e-6, **cfg):
    """Train, build and verify one config; the learning rate decays exponentially from lr to lr_final."""
    cfg['epochs'] = max(20, int(cfg['epochs'] * EPOCH_SCALE))
    RX.SEED = Path(seed) if seed and Path(seed).exists() else SORTED_SEED
    print(f'=== {name}: seed {RX.SEED.name}, {cfg}')
    TQ.Train = functools.partial(TS.Train, learning_rate_final=lr_final)
    try:
        RX.run(name, cfg, REF)
    except Exception:
        traceback.print_exc()
    package()

## Starting points

In [ ]:
# Eigendecomposition-based starts for (8, 143, C), built from the FCIDUMP (about a minute each)
import df_init

BASE_INIT = seed_npz('init_8_143_44', search_root=TOOLKIT_SRC)   # supplies H1_I and Hsym_lower


def make_init(rank, bases, copies):
    """Write seeds/init_<R>_<B>_<C>.npz with df_init.initialize and return its path."""
    path = WORK / 'seeds' / f'init_{rank}_{bases}_{copies}.npz'
    if not path.exists():
        base = np.load(BASE_INIT)
        _, _, _, S, _ = RX.lab.inputs(BASE_INIT)
        U, W = df_init.initialize(REF['tei'], S, rank, bases, copies)
        np.savez(path, U=U, W=W, H2_I=W.sum(axis=1), H1_I=base['H1_I'], Hsym_lower=base['Hsym_lower'])
    print('init:', path)
    return path


def two_stages(tag, copies):
    """Stage 1 and stage 2 for (8, 143, copies); returns the stage 2 factorization path."""
    seed = make_init(8, 143, copies)
    train(f'{tag}_stage1', seed=seed, rank=8, bases=143, copies=copies, epochs=100_000,
          lr=1e-3, lr_final=1e-5, rho=1e-4, epsilon_dfthc=0.012)
    s1 = WORK / 'runs' / f'{tag}_stage1.npz'
    train(tag, seed=s1 if s1.exists() else seed, rank=8, bases=143, copies=copies, epochs=200_000,
          lr=1e-4, lr_final=1e-6, rho=1e-4, epsilon_dfthc=0.012)
    return WORK / 'runs' / f'{tag}.npz'


def qat_if_needed(tag, source, brot=12):
    """Weight refit at `brot` bits, only when the plain build did not pass."""
    rows = [json.loads(line) for line in open(WORK / 'results.jsonl')] if (WORK / 'results.jsonl').exists() else []
    if source.exists() and not any(r['name'] == tag and r.get('status') == 'PASS' for r in rows):
        run_qat(f'{tag}_q{brot}', source, brot=brot)


In [ ]:
def warm_seed(name):
    """Path of a factorization `name` from the attached inputs (e.g. a previous version's output), or None."""
    try:
        return seed_npz(name, search_root=INPUT)
    except FileNotFoundError:
        print(f'{name} not attached: starting from scratch')
        return None


def expand_copies(source, copies, scale=1e-3):
    """Pad a factorization to `copies` copies with tiny random weights, keeping its error unchanged.

    The kit's own padding draws the new weights at 0.01, which throws a 48 mHa start to ~420 mHa.
    """
    z = dict(np.load(source))
    R, B, C0 = z['W'].shape
    if copies <= C0:
        return source
    rng = np.random.default_rng(0)
    new = scale * rng.standard_normal((R, B, copies - C0))
    z['W'] = np.concatenate([z['W'], new], axis=2)
    z['H2_I'] = np.concatenate([z['H2_I'], new.sum(axis=1)], axis=1)   # new copies: no one-body offset
    path = WORK / 'seeds' / f'{Path(source).stem}_to_C{copies}.npz'
    np.savez(path, **{key: z[key] for key in FACTORIZATION_KEYS})
    print('expanded:', path)
    return path


def single_stage(tag, copies, seed=None, lr=1e-3):
    """One 200k-epoch run of (8, 143, copies); lr decays exponentially to 1e-6."""
    seed = seed or make_init(8, 143, copies)
    train(tag, seed=seed, rank=8, bases=143, copies=copies, epochs=200_000,
          lr=lr, lr_final=1e-6, rho=1e-4, epsilon_dfthc=0.012)
    return WORK / 'runs' / f'{tag}.npz'


## C = 143, warm-started from S100 when available

In [ ]:
S100_SEED = warm_seed('S100')
S143_SEED = expand_copies(S100_SEED, 143) if S100_SEED else None
S143 = single_stage('S143', 143, seed=S143_SEED, lr=3e-5 if S143_SEED else 1e-3)
qat_if_needed('S143', S143, brot=13)
qat_if_needed('S143', S143, brot=12)

## C = 200 from scratch

In [ ]:
S200 = single_stage('S200', 200)
qat_if_needed('S200', S200, brot=13)
qat_if_needed('S200', S200, brot=12)

## Clean up
Keeps `submission.zip`, `results.jsonl`, the trained factorizations (`runs/`) and the builds.

In [ ]:
os.chdir(WORK)
for path in (WORK / 'toolkit', WORK / 'kit', WORK / 'femoco76_ref_cache.npz', WORK / 'seeds'):
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
print(sorted(p.name for p in WORK.iterdir()))